# EMD-Transformer-BiLSTM — Prediksi AQI Jakarta
Dataset: Jakarta US_AQI (Open-Meteo CAMS + weather)
Pipeline: EMD(12 IMF) → per-IMF Transformer-BiLSTM → Linear fallback → fusi BiLSTM

**Repo & hasil disimpan di Google Drive**, bukan di `/content` — supaya tidak hilang saat runtime Colab didaur ulang (idle ~90 menit / umur VM ~12 jam).
`results/` (log, CSV, plot) dan `results/checkpoints/` ada di `.gitignore`, jadi aman terhadap `git reset --hard` saat notebook dijalankan ulang.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

REPO_URL   = "https://github.com/madajabbar/riset-aqi-jakarta_ML.git"
BRANCH     = "main"
DRIVE_REPO = "/content/drive/MyDrive/riset-aqi-jakarta_ML"

if not os.path.exists(DRIVE_REPO):
    !git clone -b {BRANCH} {REPO_URL} {DRIVE_REPO}
elif os.path.exists(f"{DRIVE_REPO}/.git"):
    # reset --hard hanya menyentuh file yang di-track; results/ (gitignored) tidak dihapus
    !git -C {DRIVE_REPO} fetch origin {BRANCH} && git -C {DRIVE_REPO} reset --hard origin/{BRANCH}

%cd {DRIVE_REPO}

# Jendela tanggal: uji cepat 1 bulan -> "2022-09-01", "2022-10-01"
START, END = "2022-08-15", "2026-10-01"


In [ ]:
import sys
!{sys.executable} -m pip install -q torch scikit-learn matplotlib pandas numpy EMD-signal


In [ ]:
# Verifikasi data + import
import os
aqi, met = "our_data/jakarta_aqi.csv", "our_data/jakarta_met.csv"
assert os.path.exists(aqi), f"{aqi} missing!"
assert os.path.exists(met), f"{met} missing!"
print(f"AQI: {os.path.getsize(aqi):,} bytes | Weather: {os.path.getsize(met):,} bytes")

from PyEMD import EMD
print("PyEMD OK")


In [ ]:
# Lengan weather (RQ2) — log tersimpan di Drive
import sys
!PYTHONUNBUFFERED=1 {sys.executable} our_model/train.py --weather --start {START} --end {END} 2>&1 | tee our_model/results/full_weather.log


In [ ]:
# Lengan base (ablasi tanpa weather)
import sys
!PYTHONUNBUFFERED=1 {sys.executable} our_model/train.py --start {START} --end {END} 2>&1 | tee our_model/results/full_base.log


In [ ]:
# Hasil
from IPython.display import Image, display
import glob, pandas as pd

for f in sorted(glob.glob("our_model/results/*.png")):
    print(f"--- {f} ---"); display(Image(filename=f))
for f in sorted(glob.glob("our_model/results/*.csv")):
    print(f"\n{f}:"); print(pd.read_csv(f, header=None))
